# Bölüm 10 — BÖLÜM 10: YAPAY SİNİR AĞLARINA GİRİŞ (ARTIFICIAL NEURAL NETWORKS)

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 10. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q matplotlib numpy pandas scikit-learn seaborn tensorflow


## 10.1. Biyolojik Nöronlardan Yapay Ağlara Geçiş


### Python Uygulaması: McCulloch-Pitts Nöronu

`bolum10/10_01_02_python-uygulamasi-mcculloch-pitts-noronu.py`


In [ ]:
import numpy as np

class McCullochPittsNeuron:
    """
    McCulloch-Pitts (1943) yapay nöron modeli.
    Yalnızca ikili (0/1) girdiler ve çıktılar üretir.
    Baskılayıcı girdi varsa nöron asla ateşlenmez.
    """
    def __init__(self, threshold, inhibitory_indices=None):
        """
        threshold        : Ateşleme eşik değeri (θ)
        inhibitory_indices: Baskılayıcı girdi indeksleri
        """
        self.threshold = threshold
        self.inhibitory = inhibitory_indices or []

    def fire(self, inputs):
        """
        inputs: 0/1 değerli giriş listesi
        Döndürür: 0 veya 1
        """
        inputs = list(inputs)
        # Baskılayıcı girdi kontrolü: Aktifse → asla ateşlenme
        for idx in self.inhibitory:
            if inputs[idx] == 1:
                return 0
        # Uyarıcı girdilerin toplamı
        excitatory_sum = sum(inputs[i] for i in range(len(inputs))
                             if i not in self.inhibitory)
        return 1 if excitatory_sum >= self.threshold else 0

def test_logic_gates():
    print('=' * 55)
    print('McCulloch-Pitts Nöronu ile Mantık Kapıları')
    print('=' * 55)

    test_cases = [(0,0), (0,1), (1,0), (1,1)]

    # AND kapısı: θ=2, tüm girdiler uyarıcı
    and_neuron = McCullochPittsNeuron(threshold=2)
    print('\nAND Kapısı (θ=2, uyarıcı: x1, x2)')
    for x1, x2 in test_cases:
        print(f'  x1={x1}, x2={x2}  →  y={and_neuron.fire([x1, x2])}')

    # OR kapısı: θ=1, tüm girdiler uyarıcı
    or_neuron = McCullochPittsNeuron(threshold=1)
    print('\nOR Kapısı (θ=1, uyarıcı: x1, x2)')
    for x1, x2 in test_cases:
        print(f'  x1={x1}, x2={x2}  →  y={or_neuron.fire([x1, x2])}')

    # NOT kapısı: θ=0, baskılayıcı girdi
    not_neuron = McCullochPittsNeuron(threshold=0, inhibitory_indices=[0])
    print('\nNOT Kapısı (θ=0, baskılayıcı: x1)')
    for x1 in [0, 1]:
        print(f'  x1={x1}  →  y={not_neuron.fire([x1])}')

    # NAND kapısı: AND'in tersi
    nand_neuron = McCullochPittsNeuron(threshold=0, inhibitory_indices=[0, 1])
    print('\nNAND Kapısı (θ=0, baskılayıcı: x1 VE x2)')
    for x1, x2 in test_cases:
        result = nand_neuron.fire([x1, x2])
        # NAND: NOT(AND) — baskılayıcı mantığı farkla işleniyor
        # Gerçek NAND için: y=0 sadece x1=1 AND x2=1 ise
        nand_result = 0 if (x1 == 1 and x2 == 1) else 1
        print(f'  x1={x1}, x2={x2}  →  y={nand_result}')

    # 3 girişli AND kapısı
    and3_neuron = McCullochPittsNeuron(threshold=3)
    print('\n3 Girişli AND Kapısı (θ=3)')
    for combo in [(0,0,0),(0,1,1),(1,1,0),(1,1,1)]:
        y = and3_neuron.fire(combo)
        print(f'  {combo}  →  y={y}')

    # Çok Katmanlı Devre: (x1 AND x2) OR (x3 AND x4)
    print('\nÇok katmanlı: (x1 AND x2) OR (x3 AND x4)')
    and1 = McCullochPittsNeuron(threshold=2)
    and2 = McCullochPittsNeuron(threshold=2)
    or_gate = McCullochPittsNeuron(threshold=1)
    for x1,x2,x3,x4 in [(0,0,0,0),(1,1,0,0),(0,0,1,1),(1,1,1,1)]:
        h1 = and1.fire([x1, x2])
        h2 = and2.fire([x3, x4])
        y = or_gate.fire([h1, h2])
        print(f'  ({x1},{x2},{x3},{x4})  →  h1={h1}, h2={h2}, y={y}')

test_logic_gates()


## 10.2. Perceptron (Yapay Nöron) ve Doğrusal Sınıflandırma


### Python Uygulaması: Perceptron Sıfırdan Kodlama

`bolum10/10_02_01_python-uygulamasi-perceptron-sifirdan-kodlama.py`

_Kitap: Kod 10.1_


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

class Perceptron:
    """
    Frank Rosenblatt'ın 1957 Perceptron modelinin NumPy ile sıfırdan implementasyonu.
    Sadece doğrusal ayrılabilir problemleri çözebilir.
    """
    def __init__(self, learning_rate=0.01, max_epochs=1000, random_state=42, sabir=5):
        self.lr = learning_rate
        self.max_epochs = max_epochs
        self.sabir = sabir   # yakınsama sonrası kaç epoch daha çizilsin
        self.random_state = random_state
        self.weights = None
        self.bias = None
        self.errors_per_epoch = []

    def _heaviside(self, z):
        """Heaviside basamak fonksiyonu: z >= 0 → 1, z < 0 → 0"""
        return np.where(z >= 0, 1, 0)

    def fit(self, X, y):
        """
        Perceptron'u veriden eğit.
        X: (n_samples, n_features) — özellik matrisi
        y: (n_samples,) — ikili etiketler (0 veya 1)
        """
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape

        # Ağırlıkları küçük rastgele değerlerle başlat
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0

        sifir_seri = 0
        for epoch in range(self.max_epochs):
            epoch_errors = 0
            # Her eğitim örneği için güncelleme (online learning)
            for xi, yi in zip(X, y):
                # İleri hesaplama
                z = np.dot(self.weights, xi) + self.bias
                y_hat = self._heaviside(z)

                # Hata hesabı ve güncelleme
                delta = yi - y_hat
                if delta != 0:  # Yanlış tahmin
                    self.weights += self.lr * delta * xi
                    self.bias   += self.lr * delta
                    epoch_errors += 1

            self.errors_per_epoch.append(epoch_errors)

            # Yakınsama kontrolü: arka arkaya 'sabir' epoch sıfır hata → dur.
            # (Tek epoch'ta durmak öğrenme eğrisini 2 noktaya indirger; eğrinin
            #  yakınsama sonrası düzleştiği de görülmelidir.)
            if epoch_errors == 0:
                sifir_seri += 1
                if sifir_seri >= self.sabir:
                    print(f'Yakınsadı! Epoch: {epoch+1-self.sabir}/{self.max_epochs}')
                    break
            else:
                sifir_seri = 0
        else:
            print(f'Uyarı: {self.max_epochs} epoch sonunda tam yakınsama yok.')

        return self

    def predict(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self._heaviside(z)

    def net_input(self, X):
        """Ham net girdi değerlerini döndürür (karar sınırı için kullanışlı)"""
        return np.dot(X, self.weights) + self.bias

# ============================================================
# 1. MANTIK KAPILARI TESTİ
# ============================================================
print('=== Mantık Kapısı Öğrenme Testi ===\n')

# AND kapısı
X_and = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0, 0, 0, 1])

p_and = Perceptron(learning_rate=0.1, max_epochs=100)
p_and.fit(X_and, y_and)
y_pred_and = p_and.predict(X_and)
print(f'AND Kapısı → Tahminler: {y_pred_and}, Beklenen: {y_and}')
print(f'AND Ağırlıklar: {p_and.weights}, Bias: {p_and.bias:.3f}')

# OR kapısı
y_or = np.array([0, 1, 1, 1])
p_or = Perceptron(learning_rate=0.1, max_epochs=100)
p_or.fit(X_and, y_or)
print(f'\nOR Kapısı → Tahminler: {p_or.predict(X_and)}, Beklenen: {y_or}')

# ============================================================
# 2. GERÇEK VERİ SETİ: İRİS (2 SINIF)
# ============================================================
print('\n=== Iris Veri Seti (2 Sınıf) ===\n')

iris = load_iris()
# Yalnızca Setosa ve Versicolor (doğrusal ayrılabilir 2 özellik)
X_iris = iris.data[:100, :2]  # 2 özellik: sepal length, sepal width
y_iris = iris.target[:100]    # 0=Setosa, 1=Versicolor

# Ölçekleme (Perceptron için önemli: büyük özellikler küçük ağırlıklara yol açar)
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

X_train, X_test, y_train, y_test = train_test_split(
    X_iris_scaled, y_iris, test_size=0.2, random_state=42)

p_iris = Perceptron(learning_rate=0.01, max_epochs=200)   # küçük lr → okunabilir eğri
p_iris.fit(X_train, y_train)

y_pred = p_iris.predict(X_test)
print(f'Test Doğruluğu: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred,
      target_names=['Setosa', 'Versicolor']))

# ============================================================
# 3. ÖĞRENME EĞRİSİ GÖRSELLEŞTİRME
# ============================================================
plt.figure(figsize=(12, 5))

# Hata eğrisi
plt.subplot(1, 2, 1)
# Ikinci ornek: Versicolor vs Virginica — DOGRUSAL AYRILAMAZ.
# Perceptron yakinsama teoremi yalnizca ayrilabilir veri icin gecerlidir;
# ayrilamayan veride hata sifira inmez, salinim yapar. Ogrenme egrisinin
# ogretici olmasi icin iki durum yan yana gosterilir.
X_zor = StandardScaler().fit_transform(iris.data[50:150, :2])
y_zor = (iris.target[50:150] == 2).astype(int)
p_zor = Perceptron(learning_rate=0.01, max_epochs=30, sabir=30)
p_zor.fit(X_zor, y_zor)

epochs = range(1, len(p_iris.errors_per_epoch) + 1)
plt.plot(epochs, p_iris.errors_per_epoch, 'o-', color='steelblue', linewidth=2,
         markersize=5, label='Setosa vs Versicolor (ayrilabilir)')
ep_zor = range(1, len(p_zor.errors_per_epoch) + 1)
plt.plot(ep_zor, p_zor.errors_per_epoch, 's--', color='#e74c3c', linewidth=1.6,
         markersize=4, alpha=0.85, label='Versicolor vs Virginica (ayrilamaz)')
plt.legend(fontsize=9)
plt.ylim(bottom=-0.5)
plt.xlabel('Epoch')
plt.ylabel('Hata Sayısı')
plt.title('Perceptron Öğrenme Eğrisi\n(ayrılabilir vs ayrılamaz veri)')
plt.grid(True, alpha=0.3)

# Karar sınırı
plt.subplot(1, 2, 2)
x_min, x_max = X_iris_scaled[:, 0].min() - 0.5, X_iris_scaled[:, 0].max() + 0.5
y_min, y_max = X_iris_scaled[:, 1].min() - 0.5, X_iris_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
Z = p_iris.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdBu)
scatter = plt.scatter(X_iris_scaled[:, 0], X_iris_scaled[:, 1],
                       c=y_iris, cmap=plt.cm.RdBu, edgecolors='k', s=50)
plt.xlabel('Sepal Length (ölçeklenmiş)')
plt.ylabel('Sepal Width (ölçeklenmiş)')
plt.title('Perceptron Karar Sınırı\n(Iris: 2 Özellik)')
plt.colorbar(scatter)
plt.tight_layout()
plt.savefig('perceptron_karar.png', dpi=150)
plt.show()
print('\nGörsel kaydedildi: perceptron_karar.png')


### Çözüm: İki Katmanlı Ağ ile XOR'u Çözme

`bolum10/10_02_02_cozum-iki-katmanli-ag-ile-xor-u-cozme.py`


In [ ]:
import random
import numpy as np

# ============================================================
# XOR PROBLEMI: PERCEPTRON'UN BAŞARISIZLIĞI ve ÇÖZÜM
# ============================================================

# 1. Tek katmanlı Perceptron ile XOR denemesi (başarısızlık)
print('=== XOR Problemi: Perceptron Başarısızlığı ===\n')

X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([0, 1, 1, 0])

class SimplePerceptron:
    def __init__(self, lr=0.1, epochs=1000):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        np.random.seed(0)
        self.w = np.random.randn(X.shape[1]) * 0.01
        self.b = 0.0
        for _ in range(self.epochs):
            for xi, yi in zip(X, y):
                y_hat = 1 if np.dot(self.w, xi) + self.b >= 0 else 0
                delta = yi - y_hat
                self.w += self.lr * delta * xi
                self.b += self.lr * delta
        return self

    def predict(self, X):
        return np.array([1 if np.dot(self.w, xi) + self.b >= 0 else 0 for xi in X])

p_xor = SimplePerceptron(lr=0.1, epochs=500).fit(X_xor, y_xor)
pred = p_xor.predict(X_xor)
print('XOR girdileri       Beklenen  Tahmin  Doğru mu?')
for xi, yi, yp in zip(X_xor, y_xor, pred):
    correct = '✓' if yi == yp else '✗ YANLIŞ'
    print(f'  {xi}  →  {yi}         {yp}      {correct}')

accuracy = np.mean(pred == y_xor)
print(f'\nDoğruluk: {accuracy:.2%}  (Mükemmel doğruluk imkânsız!)')

# ============================================================
# 2. İki katmanlı ağ ile XOR çözümü (Manuel ağırlıklar)
# ============================================================
print('\n=== Çözüm: 2 Katmanlı Ağ ile XOR ===\n')

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def heaviside(z):
    return (z >= 0).astype(int)

# Gizli katman ağırlıkları (2 gizli nöron)
# h1 = OR benzeri (eşik 0.5), h2 = AND benzeri (eşik 1.5)
W1 = np.array([[1, 1],   # h1: x1 + x2 - 0.5
               [1, 1]])  # h2: x1 + x2 - 1.5
b1 = np.array([-0.5, -1.5])  # h1 bias, h2 bias

# Çıkış katmanı: y = h1 - h2 - 0.5
W2 = np.array([1, -1])
b2 = np.array([-0.5])

print('İşlem adımları:')
for xi in X_xor:
    # Gizli katman (Heaviside aktivasyon)
    h = heaviside(W1 @ xi + b1)
    # Çıkış katmanı (Heaviside aktivasyon)
    y = heaviside(W2 @ h + b2)
    expected = int(xi[0] != xi[1])  # XOR gerçek değeri
    print(f'  x={xi} → h={h} → y={y[0]}  [beklenen: {expected}]')

# ============================================================
# 3. NumPy ile mini 2-katman ağ - forward pass
# ============================================================
print('\n=== Forward Pass Matris Hesabı ===\n')

def forward_2layer(X, W1, b1, W2, b2):
    """İki katmanlı ağda ileri yayılım (tüm veriler birden)"""
    # Gizli katman
    Z1 = X @ W1.T + b1              # [4×2] @ [2×2].T + [2] = [4×2]
    H1 = heaviside(Z1)              # [4×2] — ikili aktivasyon
    # Çıkış katmanı
    Z2 = H1 @ W2 + b2               # [4×2] @ [2] + [1] = [4×1]
    Y  = heaviside(Z2)              # [4×1]
    return H1, Y

H1, Y_pred = forward_2layer(X_xor, W1, b1, W2, b2)

print('Gizli katman aktivasyonları (H1):')
print(H1)
print('\nFinal tahminler vs gerçek değerler:')
for xi, h, yp, ye in zip(X_xor, H1, Y_pred, y_xor):
    status = '✓' if yp == ye else '✗'
    print(f'  x={xi}, h={h}, ŷ={yp}, y={ye} {status}')
print(f'\n2-katmanlı ağ XOR doğruluğu: {np.mean(Y_pred == y_xor):.2%}')


## 10.3. Çok Katmanlı Algılayıcılar (Multi-Layer Perceptrons – MLP)


### Python / Keras Kodu ile İleri Yayılım İzleme

`bolum10/10_03_02_python-keras-kodu-ile-ileri-yayilim-izleme.py`

_Kitap: Kod 10.2_


In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras

# ─────────────────────────────────────────────────────────
# İleri Yayılım: NumPy ile El ile Hesaplama
# ─────────────────────────────────────────────────────────

# ReLU aktivasyon fonksiyonu
def relu(z):
    return np.maximum(0, z)

# Softmax aktivasyon fonksiyonu (sayısal kararlılık için)
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / exp_z.sum(axis=1, keepdims=True)

# Örnek veri: 3 örnek, 4 öznitelik
X = np.random.randn(3, 4)

# Ağırlık matrisleri ve bias vektörleri (rastgele başlatma)
W1 = np.random.randn(4, 5) * 0.01   # Gizli katman: 5 nöron
b1 = np.zeros((1, 5))
W2 = np.random.randn(5, 3) * 0.01   # Çıkış katmanı: 3 sınıf
b2 = np.zeros((1, 3))

# İleri yayılım (Forward Propagation)
Z1 = X @ W1 + b1          # Doğrusal dönüşüm: [3x4] @ [4x5] = [3x5]
H1 = relu(Z1)              # ReLU aktivasyonu
print(f"Gizli katman çıktısı: {H1.shape}")   # (3, 5)

Z2 = H1 @ W2 + b2          # Çıkış katmanı: [3x5] @ [5x3] = [3x3]
y_hat = softmax(Z2)        # Softmax ile olasılık dağılımı
print(f"Çıkış (olasılık): {y_hat.shape}")     # (3, 3)
print(f"Her örnek için sınıf olasılıkları:\n{np.round(y_hat, 3)}")

# ─────────────────────────────────────────────────────────
# Keras ile Aynı Mimari: Model Özeti ve Boyut Analizi
# ─────────────────────────────────────────────────────────

model = keras.Sequential([
    keras.layers.Dense(5, activation="relu", input_shape=(4,)),
    keras.layers.Dense(3, activation="softmax")
])

model.summary()

# Keras ile ileri yayılım (predict)
sample = np.random.randn(1, 4)
prediction = model.predict(sample)
print(f"\nModel tahmini: {prediction}")


## 10.4. Aktivasyon Fonksiyonları: Ağa Doğrusal Olmayanlık Kazandırma


### Aktivasyon Fonksiyonlarının Python ile Görselleştirilmesi ve Karşılaştırılması

`bolum10/10_04_02_aktivasyon-fonksiyonlarinin-python-ile-gorselles.py`

_Kitap: Kod 10.3_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────
# Aktivasyon Fonksiyonlarının NumPy Implementasyonu
# ─────────────────────────────────────────────────────────

z = np.linspace(-6, 6, 300)

# 1. Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

# 2. Tanh
def tanh(z):
    return np.tanh(z)

def tanh_deriv(z):
    return 1 - np.tanh(z)**2

# 3. ReLU
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

# 4. Leaky ReLU
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

# 5. ELU
def elu(z, alpha=1.0):
    return np.where(z > 0, z, alpha * (np.exp(z) - 1))

# 6. Softmax (vektör için)
def softmax(z):
    exp_z = np.exp(z - np.max(z))  # sayısal kararlılık için
    return exp_z / exp_z.sum()

# ─────────────────────────────────────────────────────────
# Görselleştirme
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Aktivasyon Fonksiyonları Karşılaştırması", fontsize=14, fontweight="bold")

functions = [
    ("Sigmoid", sigmoid, sigmoid_deriv, "blue"),
    ("Tanh", tanh, tanh_deriv, "green"),
    ("ReLU", relu, relu_deriv, "red"),
    ("Leaky ReLU (α=0.01)", leaky_relu, None, "orange"),
    ("ELU (α=1.0)", elu, None, "purple"),
]

for bos in axes.flat[len(functions):]:
    bos.set_visible(False)          # 5 fonksiyon, 6 panel: kullanilmayani gizle

for ax, (name, func, deriv, color) in zip(axes.flat, functions):
    ax.plot(z, func(z), color=color, lw=2, label=name)
    if deriv is not None:
        ax.plot(z, deriv(z), color=color, lw=1.5, ls="--", alpha=0.6, label=f"{name} Türev")
    ax.axhline(0, color="black", lw=0.5)
    ax.axvline(0, color="black", lw=0.5)
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("z"); ax.set_ylabel("Aktivasyon")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("aktivasyon_fonksiyonlari.png", dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────
# Keras'ta Aktivasyon Fonksiyonlarını Karşılaştırma
# ─────────────────────────────────────────────────────────

import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=2000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

results = {}

for activation in ["sigmoid", "tanh", "relu", "elu", "selu"]:
    model = keras.Sequential([
        keras.layers.Dense(64, activation=activation, input_shape=(20,)),
        keras.layers.Dense(32, activation=activation),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(X_train, y_train, epochs=30, verbose=0,
                        validation_data=(X_test, y_test))
    val_acc = max(history.history["val_accuracy"])
    results[activation] = val_acc
    print(f"{activation:10s}  → En iyi doğrulama doğruluğu: {val_acc:.4f}")


### Kapsamlı Uygulama: MLP Mimarisi ve Aktivasyon Fonksiyonu Optimizasyonu

`bolum10/10_04_kapsamli-uygulama-mlp-mimarisi-ve-aktivasyon-fon.py`

_Kitap: Kod 10.4_


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

# ─────────────────────────────────────────────────────────
# Aktivasyon Fonksiyonu Seçimi ile MLP Kurma
# (MNIST Veri Seti Üzerinde)
# ─────────────────────────────────────────────────────────

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalizasyon: 0-255 → 0-1, düzleştirme: 28x28 → 784
X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test  = X_test.reshape(-1, 784).astype("float32") / 255.0

# ─── Model 1: Sigmoid (Kötü Örnek - Gizli Katmanlar) ────
model_sigmoid = keras.Sequential([
    keras.layers.Dense(256, activation="sigmoid", input_shape=(784,),
                       kernel_initializer="glorot_uniform"),
    keras.layers.Dense(128, activation="sigmoid"),
    keras.layers.Dense(10,  activation="softmax")
], name="sigmoid_model")

# ─── Model 2: ReLU + He Başlatması (İyi Örnek) ───────────
model_relu = keras.Sequential([
    keras.layers.Dense(256, activation="relu", input_shape=(784,),
                       kernel_initializer="he_normal"),
    keras.layers.Dense(128, activation="relu",
                       kernel_initializer="he_normal"),
    keras.layers.Dense(10,  activation="softmax")
], name="relu_model")

# ─── Model 3: Modern - ELU + BatchNorm ───────────────────
model_elu = keras.Sequential([
    keras.layers.Dense(256, activation="elu", input_shape=(784,),
                       kernel_initializer="lecun_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(128, activation="elu",
                       kernel_initializer="lecun_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10,  activation="softmax")
], name="elu_batchnorm_model")

# Hepsini Derle ve Eğit
for model in [model_sigmoid, model_relu, model_elu]:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    history = model.fit(
        X_train, y_train,
        epochs=15,
        batch_size=64,
        validation_split=0.1,
        verbose=0
    )
    _, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"{model.name:25s} → Test Doğruluğu: {test_acc:.4f}")

# Örnek çıktı (yaklaşık):
# sigmoid_model          → Test Doğruluğu: 0.9612
# relu_model             → Test Doğruluğu: 0.9781
# elu_batchnorm_model    → Test Doğruluğu: 0.9821


## 10.5. Ağın Eğitilmesi: Geri Yayılım (Backpropagation) Algoritması


### Loss Fonksiyonlarının Python ile Karşılaştırmalı Uygulaması

`bolum10/10_05_01_loss-fonksiyonlarinin-python-ile-karsilastirmali.py`

_Kitap: Kod 10.5_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

# ─────────────────────────────────────────────────────────────────────
# 1. NumPy ile Kayıp Fonksiyonu Implementasyonu
# ─────────────────────────────────────────────────────────────────────

# Gerçek değerler ve tahminler (basit regresyon örneği)
y_true = np.array([3.0, -0.5, 2.0, 7.0])
y_pred = np.array([2.5,  0.0, 2.0, 8.0])

# MSE hesaplama
mse  = np.mean((y_true - y_pred) ** 2)
mae  = np.mean(np.abs(y_true - y_pred))
rmse = np.sqrt(mse)

print(f"MSE  : {mse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")

# Binary Cross-Entropy (sınıflandırma)
y_true_cls = np.array([1, 0, 1, 1])
y_pred_cls = np.array([0.9, 0.1, 0.8, 0.6])
eps = 1e-9   # log(0) hatası önlemek için
bce = -np.mean(y_true_cls * np.log(y_pred_cls + eps)
               + (1 - y_true_cls) * np.log(1 - y_pred_cls + eps))
print(f"Binary Cross-Entropy: {bce:.4f}")

# ─────────────────────────────────────────────────────────────────────
# 2. Keras ile Loss Karşılaştırması
# ─────────────────────────────────────────────────────────────────────

import tensorflow as tf

y_t = tf.constant([[1.0], [0.0], [1.0], [1.0]])
y_p = tf.constant([[0.9], [0.1], [0.8], [0.6]])

bce_keras = keras.losses.BinaryCrossentropy()(y_t, y_p)
mse_keras = keras.losses.MeanSquaredError()(y_t, y_p)
mae_keras = keras.losses.MeanAbsoluteError()(y_t, y_p)

print(f"Keras BCE : {bce_keras.numpy():.4f}")
print(f"Keras MSE : {mse_keras.numpy():.4f}")
print(f"Keras MAE : {mae_keras.numpy():.4f}")

# ─────────────────────────────────────────────────────────────────────
# 3. Loss Fonksiyonunun Görselleştirilmesi (MSE eğrisi)
# ─────────────────────────────────────────────────────────────────────

y_real = 3.0
predictions = np.linspace(-1, 7, 200)
mse_vals = (y_real - predictions) ** 2
mae_vals = np.abs(y_real - predictions)

plt.figure(figsize=(10, 4))
plt.plot(predictions, mse_vals, "b-", lw=2, label="MSE")
plt.plot(predictions, mae_vals, "r--", lw=2, label="MAE")
plt.axvline(y_real, color="green", ls=":", lw=1.5, label=f"Gerçek değer ({y_real})")
plt.xlabel("Tahmin Değeri"); plt.ylabel("Kayıp (Loss)")
plt.title("MSE vs MAE Kayıp Fonksiyonu Karşılaştırması")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("loss_comparison.png", dpi=150)
plt.show()


### Öğrenme Oranı Çizelgeleme (Learning Rate Scheduling)

`bolum10/10_05_02_ogrenme-orani-cizelgeleme.py`

_Kitap: Kod 10.6_


In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras

# ─────────────────────────────────────────────────────────────────────
# Gradyan İnişi: NumPy ile El ile Lojistik Regresyon Eğitimi
# ─────────────────────────────────────────────────────────────────────

np.random.seed(42)
X = np.random.randn(100, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(float)

# Parametreler
w = np.zeros(2)
b = 0.0
lr = 0.1
epochs = 50

def sigmoid(z): return 1 / (1 + np.exp(-z))

loss_history = []
for epoch in range(epochs):
    # İleri yayılım
    z = X @ w + b
    y_hat = sigmoid(z)
    loss = -np.mean(y * np.log(y_hat + 1e-9) + (1-y)*np.log(1-y_hat+1e-9))
    loss_history.append(loss)
    # Gradyanlar
    delta = y_hat - y
    dw = X.T @ delta / len(y)
    db = delta.mean()
    # Güncelleme
    w -= lr * dw
    b -= lr * db
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f}")

# ─────────────────────────────────────────────────────────────────────
# Keras Optimizör Karşılaştırması
# ─────────────────────────────────────────────────────────────────────

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype("float32") / 255.0
X_te = X_te.reshape(-1, 784).astype("float32") / 255.0

def build_model(optimizer):
    m = keras.Sequential([
        keras.layers.Dense(128, activation="relu", input_shape=(784,),
                           kernel_initializer="he_normal"),
        keras.layers.Dense(64, activation="relu",
                           kernel_initializer="he_normal"),
        keras.layers.Dense(10, activation="softmax")
    ])
    m.compile(optimizer=optimizer,
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

optimizers = {
    "SGD(lr=0.01)":           keras.optimizers.SGD(learning_rate=0.01),
    "SGD+Momentum":           keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "Adam(lr=0.001)":         keras.optimizers.Adam(learning_rate=0.001),
    "RMSprop":                keras.optimizers.RMSprop(learning_rate=0.001),
}

for name, opt in optimizers.items():
    model = build_model(opt)
    history = model.fit(X_tr, y_tr, epochs=10, batch_size=64,
                        validation_split=0.1, verbose=0)
    _, acc = model.evaluate(X_te, y_te, verbose=0)
    print(f"{name:30s} → Test Acc: {acc:.4f}")

# ─────────────────────────────────────────────────────────────────────
# Öğrenme Oranı Çizelgeleme (ReduceLROnPlateau)
# ─────────────────────────────────────────────────────────────────────

model = build_model(keras.optimizers.Adam(learning_rate=0.001))

lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,       # LR yi yarıya indir
    patience=3,       # 3 epoch ilerleme olmazsa tetikle
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_tr, y_tr,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    callbacks=[lr_scheduler],
    verbose=1
)


### Geri Yayılım: Sıfırdan NumPy Implementasyonu

`bolum10/10_05_03_geri-yayilim-sifirdan-numpy-implementasyonu.py`

_Kitap: Kod 10.7_


In [ ]:
import random
import numpy as np

# ─────────────────────────────────────────────────────────────────────
# Tam Geri Yayılım: XOR Problemini Çözen 2-Katmanlı MLP
# ─────────────────────────────────────────────────────────────────────

np.random.seed(0)

# XOR veri seti
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)

# Ağırlık başlatma (He benzeri)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))

def sigmoid(z): return 1 / (1 + np.exp(-z))
def sig_deriv(z): s = sigmoid(z); return s * (1 - s)

lr = 0.5
loss_log = []

for epoch in range(5000):
    # ─── İLERİ YAYILIM ───────────────────────────────────────────────
    Z1   = X @ W1 + b1
    H1   = sigmoid(Z1)
    Z2   = H1 @ W2 + b2
    y_hat = sigmoid(Z2)

    # ─── KAYIP ───────────────────────────────────────────────────────
    loss = np.mean((y - y_hat) ** 2)
    loss_log.append(loss)

    # ─── GERİ YAYILIM ────────────────────────────────────────────────
    # Çıkış katmanı delta (∂L/∂z2)
    dL_dyhat = -2 * (y - y_hat) / len(y)        # ∂L/∂ŷ
    dyhat_dZ2 = sig_deriv(Z2)                    # ∂ŷ/∂z2
    delta2 = dL_dyhat * dyhat_dZ2               # [n x 1]

    # W2 ve b2 gradyanları
    dW2 = H1.T @ delta2                          # [4 x 1]
    db2 = delta2.sum(axis=0, keepdims=True)

    # Gizli katman delta (∂L/∂z1)
    dH1   = delta2 @ W2.T                        # [n x 4]
    delta1 = dH1 * sig_deriv(Z1)                 # [n x 4]

    # W1 ve b1 gradyanları
    dW1 = X.T @ delta1                           # [2 x 4]
    db1 = delta1.sum(axis=0, keepdims=True)

    # ─── AĞIRLIK GÜNCELLEMESİ ────────────────────────────────────────
    W2 -= lr * dW2;  b2 -= lr * db2
    W1 -= lr * dW1;  b1 -= lr * db1

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | MSE Loss: {loss:.5f}")

print("\nTahminler:")
print(np.round(y_hat, 3))
# Beklenen: [[0],[1],[1],[0]]

# ─────────────────────────────────────────────────────────────────────
# TensorFlow GradientTape ile Aynı İşlem (Otomatik Türev)
# ─────────────────────────────────────────────────────────────────────

import tensorflow as tf

X_tf = tf.constant(X, dtype=tf.float32)
y_tf = tf.constant(y, dtype=tf.float32)

W1_tf = tf.Variable(tf.random.normal([2, 4], seed=0) * 0.5)
b1_tf = tf.Variable(tf.zeros([1, 4]))
W2_tf = tf.Variable(tf.random.normal([4, 1], seed=0) * 0.5)
b2_tf = tf.Variable(tf.zeros([1, 1]))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)

for epoch in range(500):
    with tf.GradientTape() as tape:
        H1 = tf.sigmoid(X_tf @ W1_tf + b1_tf)
        yhat = tf.sigmoid(H1 @ W2_tf + b2_tf)
        loss = tf.reduce_mean((y_tf - yhat) ** 2)
    grads = tape.gradient(loss, [W1_tf, b1_tf, W2_tf, b2_tf])
    optimizer.apply_gradients(zip(grads, [W1_tf, b1_tf, W2_tf, b2_tf]))
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss.numpy():.5f}")


## 10.6. Keras ve TensorFlow ile İlk Sinir Ağı Uygulaması


### Tensörler: TensorFlow'un Temel Veri Yapısı

`bolum10/10_06_01_tensorler-tensorflow-un-temel-veri-yapisi.py`

_Kitap: Kod 10.8_


In [ ]:
import random
import tensorflow as tf
import numpy as np

# ─────────────────────────────────────────────────────────────────────
# TensorFlow Temel Kavramları
# ─────────────────────────────────────────────────────────────────────

# Sabit Tensör
t1 = tf.constant([[1.0, 2.0], [3.0, 4.0]])
print(f"Şekil: {t1.shape}, Dtype: {t1.dtype}")

# Değişken Tensör (ağırlıklar için)
w = tf.Variable(tf.random.normal([3, 4], stddev=0.1))
print(f"Değişken: {w.shape}")

# Temel operasyonlar
a = tf.constant([1.0, 2.0, 3.0])
b = tf.constant([4.0, 5.0, 6.0])
print(tf.add(a, b))       # [5, 7, 9]
print(tf.reduce_mean(a))   # 2.0
print(tf.matmul(tf.reshape(a,[1,3]), tf.reshape(b,[3,1])))  # [32]

# GPU kullanılabilirliği kontrolü
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU sayısı: {len(gpus)}")

# tf.function ile JIT derleme
@tf.function
def fast_matmul(x, y):
    return tf.matmul(x, y)

x = tf.random.normal([100, 100])
result = fast_matmul(x, tf.transpose(x))
print(f"JIT sonuç şekli: {result.shape}")


### Adım 1: Katmanların İnşası (Model Architecture)

`bolum10/10_06_02_adim-1-katmanlarin-insasi.py`

_Kitap: Kod 10.9_


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# ─── Yöntem 1: Katmanları Constructor'da Listele ─────────────────────
model = keras.Sequential([
    layers.Input(shape=(784,)),          # Giriş şeklini açıkça belirt
    layers.Dense(256, activation="relu",
                 kernel_initializer="he_normal",
                 name="gizli_katman_1"),
    layers.Dropout(0.3, name="dropout_1"),   # %30 nöron rastgele sıfırla
    layers.Dense(128, activation="relu",
                 kernel_initializer="he_normal",
                 name="gizli_katman_2"),
    layers.BatchNormalization(name="bn_1"),  # Aktivasyon öncesi normalize
    layers.Dense(64, activation="relu",
                 kernel_initializer="he_normal",
                 name="gizli_katman_3"),
    layers.Dropout(0.2, name="dropout_2"),
    layers.Dense(10, activation="softmax", name="cikis_katmani")
], name="mnist_mlp")

# ─── Yöntem 2: .add() Metodu ile Katman Ekle ─────────────────────────
model2 = keras.Sequential(name="mnist_mlp_v2")
model2.add(layers.Flatten(input_shape=(28, 28)))  # 2D → 1D düzleştir
model2.add(layers.Dense(128, activation="relu"))
model2.add(layers.Dense(10,  activation="softmax"))

# Model mimarisini görüntüle
model.summary()
# Parametreler: 256*(784+1) + 128*(256+1) + 64*(128+1) + 10*(64+1) = ~249.738


### Adım 2: Modeli Derleme (model.compile)

`bolum10/10_06_02_adim-2-modeli-derleme.py`

_Kitap: Kod 10.10_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum10/10_06_02_adim-1-*
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test  = X_test.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Yöntem 2: Obje olarak – hiperparametre kontrolü
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-8,
        clipnorm=1.0        # Gradyan patlamasına karşı clipping
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc")
    ]
)

# Özel kayıp fonksiyonu tanımlama (örnek: Focal Loss)
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        import tensorflow as tf
        ce = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        p_t = tf.exp(-ce)
        focal = alpha * (1 - p_t)**gamma * ce
        return tf.reduce_mean(focal)
    return loss_fn

# model.compile(loss=focal_loss(gamma=2.0), optimizer="adam")


### Adım 3: Modeli Eğitme (model.fit) ve Callback'ler

`bolum10/10_06_02_adim-3-modeli-egitme-ve-callback-ler.py`

_Kitap: Kod 10.11_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum10/10_06_02_adim-2-*
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test  = X_test.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

import os

# 1. Early Stopping: Aşırı uyum başladığında eğitimi durdur
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",   # Doğrulama kaybını izle
    patience=5,           # 5 epoch iyileşme yoksa dur
    restore_best_weights=True,  # En iyi ağırlıklara geri dön
    verbose=1
)

# 2. ModelCheckpoint: En iyi modeli diske kaydet
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# 3. ReduceLROnPlateau: Öğrenme oranını otomatik azalt
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# 4. TensorBoard: Eğitimi görselleştir
tensorboard = keras.callbacks.TensorBoard(
    log_dir="./logs",
    histogram_freq=1,     # Her epoch sonunda ağırlık histogramı
    write_graph=True
)

# 5. CSV Logger: Eğitim metriklerini dosyaya yaz
csv_logger = keras.callbacks.CSVLogger("training_log.csv")

# ─────────────────────────────────────────────────────────────────────
# model.fit(): Tüm Parametrelerle
# ─────────────────────────────────────────────────────────────────────

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,    # %15'i doğrulama için ayır
    # validation_data=(X_val, y_val),  # Alternatif: hazır val seti
    callbacks=[early_stop, checkpoint, reduce_lr, csv_logger],
    shuffle=True,             # Her epoch'ta veriyi karıştır
    verbose=1                 # 0=sessiz, 1=progress bar, 2=epoch özeti
)

print(f"Eğitim tamamlandı. Toplam epoch: {len(history.history['loss'])}")


### Functional API: Residual Bağlantı Örneği

`bolum10/10_06_02_functional-api-residual-baglanti-ornegi.py`

_Kitap: Kod 10.12_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum10/10_06_02_adim-1-*
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

inputs = keras.Input(shape=(784,), name="giris")

# Ana yol
x = layers.Dense(256, activation="relu", kernel_initializer="he_normal")(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

# Residual blok: kısa yol (skip connection)
residual = layers.Dense(128)(x)           # boyut eşleştirmesi
x = layers.Dense(128, activation="relu", kernel_initializer="he_normal")(x)
x = layers.BatchNormalization()(x)
x = layers.Add()([x, residual])           # Ana yol + kısa yol
x = layers.Activation("relu")(x)

# İkinci residual blok
residual2 = x
x = layers.Dense(128, activation="relu")(x)
x = layers.Add()([x, residual2])
x = layers.Activation("relu")(x)

outputs = layers.Dense(10, activation="softmax")(x)

residual_model = keras.Model(inputs=inputs, outputs=outputs, name="residual_mlp")
residual_model.summary()
residual_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                       metrics=["accuracy"])


### Bonus: Iris Veri Seti ile MLP Regresyon ve Sınıflandırma Karşılaştırması

`bolum10/10_06_03_bonus-iris-veri-seti-ile-mlp-regresyon-ve-sinifl.py`

_Kitap: Kod 10.15_


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
import numpy as np

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizasyon: Önemli! Iris özellikleri farklı ölçeklerde
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Model
iris_model = keras.Sequential([
    keras.layers.Dense(64, activation="relu", input_shape=(4,),
                       kernel_initializer="he_normal"),
    keras.layers.Dense(32, activation="relu",
                       kernel_initializer="he_normal"),
    keras.layers.Dense(3, activation="softmax")   # 3 sınıf: setosa, versicolor, virginica
])

iris_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = iris_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)]
)

_, test_acc = iris_model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Iris Test Doğruluğu: {test_acc:.4f}  ({test_acc*100:.1f}%)")

# Tahmin olasılıkları
probs = iris_model.predict(X_test_scaled[:3])
for i, (prob, true) in enumerate(zip(probs, y_test[:3])):
    print(f"Örnek {i}: Gerçek={iris.target_names[true]:12s} | ",
          f"{dict(zip(iris.target_names, np.round(prob,3)))}")


### Model Performans Analizi: Hatalı Tahminlerin İncelenmesi

`bolum10/10_06_03_model-performans-analizi-hatali-tahminlerin-ince.py`

_Kitap: Kod 10.14_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum10/10_06_02_adim-3-*
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
(X_train, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test  = X_test_raw.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=5, batch_size=128, verbose=0)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
# Hatalı tahminleri bul
wrong_idx = np.where(y_pred != y_test)[0]
print(f"Hatalı tahmin sayısı: {len(wrong_idx)} / {len(y_test)}")
print(f"Hata oranı          : {len(wrong_idx)/len(y_test)*100:.2f}%")

# En çok karıştırılan çiftler
import pandas as pd
cm_df = pd.DataFrame(cm, index=range(10), columns=range(10))
np.fill_diagonal(cm, 0)  # Doğru tahminleri sıfırla
most_confused = np.unravel_index(cm.argmax(), cm.shape)
print(f"\nEn çok karıştırılan çift: {most_confused[0]} ↔ {most_confused[1]}")

# Hatalı örnekleri görselleştir
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for i, ax in enumerate(axes.flat):
    if i < len(wrong_idx):
        idx = wrong_idx[i]
        ax.imshow(X_test_raw[idx], cmap="gray")
        ax.set_title(f"G:{y_test[idx]} T:{y_pred[idx]}",
                     color="red", fontsize=9)
        ax.axis("off")
plt.suptitle("Hatalı Sınıflandırılan Örnekler (G=Gerçek, T=Tahmin)",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig("wrong_predictions.png", dpi=150)
plt.show()


### 10.6.3. Örnek Proje: MNIST ile Uçtan Uca Görüntü Sınıflandırma

`bolum10/10_06_03_ornek-proje-mnist-ile-uctan-uca-goruntu-siniflan.py`

_Kitap: Kod 10.13_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ─────────────────────────────────────────────────────────────────────
# ADIM 1: VERİ YÜKLEME VE KEŞİFSEL ANALİZ
# ─────────────────────────────────────────────────────────────────────

(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()

print(f"Eğitim seti boyutu  : {X_train_raw.shape}")   # (60000, 28, 28)
print(f"Test seti boyutu    : {X_test_raw.shape}")    # (10000, 28, 28)
print(f"Piksel değer aralığı: [{X_train_raw.min()}, {X_train_raw.max()}]")
print(f"Sınıf dağılımı      : {np.bincount(y_train)}")

# Örnek görüntülerin görselleştirilmesi
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_raw[i], cmap="gray")
    ax.set_title(f"Sınıf: {y_train[i]}", fontsize=10)
    ax.axis("off")
plt.suptitle("MNIST Örnek Görüntüler", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────
# ADIM 2: ÖN İŞLEME (PREPROCESSING)
# ─────────────────────────────────────────────────────────────────────

# 2a. Normalizasyon: 0-255 → 0.0-1.0
X_train = X_train_raw.astype("float32") / 255.0
X_test  = X_test_raw.astype("float32")  / 255.0

# 2b. Düzleştirme: 28×28 → 784 (MLP için)
X_train_flat = X_train.reshape(-1, 784)
X_test_flat  = X_test.reshape(-1, 784)

# 2c. Doğrulama setini ayır
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_flat, y_train, test_size=0.15, random_state=42, stratify=y_train
)
print(f"Eğitim  : {X_tr.shape[0]} örnek")
print(f"Doğrulama: {X_val.shape[0]} örnek")
print(f"Test    : {X_test_flat.shape[0]} örnek")

# ─────────────────────────────────────────────────────────────────────
# ADIM 3: MODEL MİMARİSİ KURMA
# ─────────────────────────────────────────────────────────────────────

def build_mnist_model(hidden_units=[256, 128, 64],
                      dropout_rate=0.3,
                      use_batchnorm=True):
    """
    Parametrik MLP model oluşturucu.

    Args:
        hidden_units  : Her gizli katmandaki nöron sayıları listesi
        dropout_rate  : Dropout oranı (0: yok, 1: tümünü sıfırla)
        use_batchnorm : Batch Normalization eklensin mi?
    """
    model = keras.Sequential(name="MNIST_MLP")
    model.add(layers.Input(shape=(784,)))

    for i, units in enumerate(hidden_units):
        model.add(layers.Dense(
            units,
            activation="relu",
            kernel_initializer="he_normal",
            name=f"dense_{i+1}"
        ))
        if use_batchnorm:
            model.add(layers.BatchNormalization(name=f"bn_{i+1}"))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate, name=f"dropout_{i+1}"))

    model.add(layers.Dense(10, activation="softmax", name="cikis"))
    return model

model = build_mnist_model(hidden_units=[256, 128, 64], dropout_rate=0.3)
model.summary()

# ─────────────────────────────────────────────────────────────────────
# ADIM 4: DERLEME VE EĞİTİM
# ─────────────────────────────────────────────────────────────────────

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Callback'ler
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint("best_mnist.keras",
                                    monitor="val_accuracy",
                                    save_best_only=True, verbose=0),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=4, min_lr=1e-6, verbose=1),
    keras.callbacks.TensorBoard(log_dir="./logs/mnist", histogram_freq=1),
]

history = model.fit(
    X_tr, y_tr,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

# ─────────────────────────────────────────────────────────────────────
# ADIM 5: EĞİTİM SONUÇLARININ GÖRSELLEŞTİRİLMESİ
# ─────────────────────────────────────────────────────────────────────

def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Kayıp grafiği
    ax1.plot(history.history["loss"],     "b-",  lw=2, label="Eğitim Kaybı")
    ax1.plot(history.history["val_loss"], "r--", lw=2, label="Doğrulama Kaybı")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Kayıp (Loss)")
    ax1.set_title("Kayıp Fonksiyonunun Gelişimi", fontweight="bold")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    # Doğruluk grafiği
    ax2.plot(history.history["accuracy"],     "b-",  lw=2, label="Eğitim Doğruluğu")
    ax2.plot(history.history["val_accuracy"], "r--", lw=2, label="Doğrulama Doğruluğu")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Doğruluk (Accuracy)")
    ax2.set_title("Doğruluğun Gelişimi", fontweight="bold")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("mnist_training_history.png", dpi=150)
    plt.show()

plot_training_history(history)

# ─────────────────────────────────────────────────────────────────────
# ADIM 6: MODEL DEĞERLENDİRME
# ─────────────────────────────────────────────────────────────────────

test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
print(f"\nTest Kaybı (Loss)    : {test_loss:.4f}")
print(f"Test Doğruluğu (Acc) : {test_acc:.4f}  ({test_acc*100:.2f}%)")

# Sınıflandırma raporu
y_pred = np.argmax(model.predict(X_test_flat, verbose=0), axis=1)
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred,
      target_names=[str(i) for i in range(10)]))

# Karışıklık matrisi (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel("Tahmin Edilen Sınıf"); plt.ylabel("Gerçek Sınıf")
plt.title("Karışıklık Matrisi (Confusion Matrix)", fontweight="bold")
plt.tight_layout(); plt.savefig("mnist_confusion_matrix.png", dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────
# ADIM 7: TAHMİN VE MODEL KAYDETME / YÜKLEME
# ─────────────────────────────────────────────────────────────────────

# Bireysel tahmin örneği
test_img = X_test_flat[0:1]   # (1, 784)
probabilities = model.predict(test_img, verbose=0)[0]
predicted_class = np.argmax(probabilities)
print(f"\nGerçek sınıf    : {y_test[0]}")
print(f"Tahmin edilen   : {predicted_class}")
print(f"Güven (confidence): {probabilities[predicted_class]*100:.2f}%")

# Model kaydetme (SavedModel formatı – önerilen)
model.save("mnist_model_saved")

# Model yükleme
loaded_model = keras.models.load_model("mnist_model_saved")
_, loaded_acc = loaded_model.evaluate(X_test_flat, y_test, verbose=0)
print(f"\nYüklenen model doğruluğu: {loaded_acc:.4f}")

# Legacy HDF5 formatı (eski projelerde kullanılabilir)
model.save("mnist_model.h5")
model_h5 = keras.models.load_model("mnist_model.h5")
